In [ ]:
# ── COLAB SETUP ───────────────────────────────────────────────────────────────
# Run this cell first on Colab to install deps, clone the repo, and fetch the
# birefnet-preprocessed dataset. No-op when running locally.
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    REPO_URL = 'https://github.com/AlessandroGhiotto/industrial-AD2L.git'
    REPO_DIR = '/content/industrial-AD2L'
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip install -q timm gdown
    DATA_DIR = 'dataset/adl-2025-2026-anomaly-detection_birefnet'
    if not os.path.exists(DATA_DIR):
        !gdown -q 'https://drive.google.com/uc?id=1lCqjjtKOhqoU3M5-4ktgiVlWnp6IFDYC' -O /content/dataset.zip
        !mkdir -p dataset && unzip -q /content/dataset.zip -d dataset/ && rm /content/dataset.zip
    print('Colab ready — CWD:', os.getcwd())


In [ ]:
# ── LOCAL SETUP ───────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore', message='No positive class found in y_true')
from pathlib import Path
sys.path.append(str(Path.cwd().parent)) # Add repo root to path
sys.path.append(str(Path.cwd()))        # Add current 'final' folder to path

from src import config_final as cfg
from src import unet_final as unet
from src import ensemble_final as ens
cfg.ensure_dirs()

print(f"Dataset Directory: {cfg.DATASET_DIR}")
print(f"Output Directory: {cfg.OUTPUT_DIR}")

# UNet Improved v2
Improvements over v1:
- **Attention gates** on U-Net skip connections
- **CutPaste** augmentation using real anomaly crops, with:
  - Geometric jitter on the defect (rotation, H/V flip, mask dilation/erosion, optional mirror-half fold)
  - Photometric jitter (brightness ±15%, contrast ±15%)
  - **Pose-aware sampling** via `pose_assignments.json` — crops are pasted onto goods of the same pose cluster; anomalous-train images get a pose label via 32×32 grayscale NN to a labeled good of the same class
  - **Foreground-aware placement**: paste positions are retried (≤8) until ≥50 % of the new mask lands on the object
  - **Feathered alpha** at the paste boundary (supervision mask stays binary)
  - **Multi-paste**: 30 % chance of a second crop being pasted on the same image
- **Fractal-noise** synthetic stain/fuzzy/localized motifs
- **TTA** (h/v flips) at inference
- **Multi-view aggregation**: cross-view max boost at test time

## Training

In [ ]:
import os

CLASS_NAME = None # Set to 'class_01' to train specific class, None for all

train_kw = dict(
    dataset_dir=str(cfg.DATASET_DIR),
    desc_csv=str(cfg.UNET_DESC_CSV),
    out_dir=str(cfg.UNET_MODEL_DIR),
    image_size=cfg.UNET_IMAGE_SIZE,
    batch_size=cfg.UNET_BATCH_SIZE,
    epochs=cfg.UNET_EPOCHS,
    lr=cfg.UNET_LR,
    val_ratio=cfg.UNET_VAL_RATIO,
    p_synth=cfg.UNET_P_SYNTH,
    p_cutpaste=cfg.UNET_P_CUTPASTE,
    p_multi_paste=cfg.UNET_P_MULTI_PASTE,
    seed=cfg.SEED,
    encoder=cfg.UNET_ENCODER,
    use_attention_gates=cfg.UNET_USE_ATTENTION_GATES,
    pose_json=str(cfg.UNET_POSE_JSON) if cfg.UNET_POSE_JSON.exists() else None,
    use_shape_bank=cfg.UNET_USE_SHAPE_BANK,
)

if CLASS_NAME is None:
    for name in sorted(os.listdir(cfg.DATASET_DIR)):
        if name.startswith('class_'):
            print(f'\n=== Training {name} ===')
            unet.train_one_class(class_name=name, **train_kw)
else:
    unet.train_one_class(class_name=CLASS_NAME, **train_kw)

## Visualization & PDF Reporting

In [ ]:
from src import visualize_final as viz
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
import torch
import os
from PIL import Image
import numpy as np

N_VIS = 10
viz_outputs = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = sorted([d for d in os.listdir(cfg.DATASET_DIR) if d.startswith('class_')])

for class_name in tqdm(classes, desc="Visualizing classes"):
    ckpt_path = cfg.UNET_MODEL_DIR / class_name / 'best.pt'
    if not ckpt_path.exists():
        continue
    model = unet.load_model(str(ckpt_path), device)

    trn_root = cfg.DATASET_DIR / class_name / 'train'
    gt_root = cfg.DATASET_DIR / class_name / 'ground_truth_train'
    examples = []
    for anom_dir in sorted(trn_root.iterdir()):
        if not anom_dir.name.startswith('anomaly_'):
            continue
        gt_dir = gt_root / anom_dir.name
        for p in sorted(anom_dir.glob('*.png')):
            mp = gt_dir / p.name
            if mp.exists():
                mask = viz.load_mask(str(mp), size=(cfg.UNET_IMAGE_SIZE, cfg.UNET_IMAGE_SIZE))
                examples.append((p, mask, anom_dir.name))
                break
        if len(examples) >= N_VIS:
            break

    for img_path, gt_mask, anom_name in examples:
        img_pil = Image.open(img_path).convert('RGB')
        img_rs = img_pil.resize((cfg.UNET_IMAGE_SIZE, cfg.UNET_IMAGE_SIZE), Image.BILINEAR)

        # Use the actual view_id from the filename (UNet was trained with view conditioning)
        _, view_id = unet._parse_view_and_id(img_path.name)
        v_t = torch.tensor([view_id], dtype=torch.long, device=device)

        # Match submission-time inference: TTA + FG masking (kills background drift)
        prob = unet._infer_single_tta(model, img_rs, v_t, device, use_tta=cfg.UNET_USE_TTA)
        fg = unet.foreground_from_image(img_rs)[0].numpy()
        prob = prob * fg

        if gt_mask.shape != prob.shape:
            print(f"WARNING: Shape mismatch for {class_name}/{anom_name}; resizing GT.")
            gt_mask = np.array(Image.fromarray((gt_mask * 255).astype(np.uint8)).resize(
                (prob.shape[1], prob.shape[0]), Image.NEAREST))
            gt_mask = (gt_mask > 0).astype(np.float32)

        ap = average_precision_score(gt_mask.ravel().astype(int), prob.ravel())
        viz_outputs.append({
            'input': img_pil,
            'gt': gt_mask,
            'heatmap': prob,
            'class_name': class_name,
            'anomaly_name': anom_name,
            'ap': ap,
        })

pdf_path = cfg.OUTPUT_DIR / 'unet_visualizations.pdf'
viz.export_to_pdf(viz_outputs, str(pdf_path), title_prefix="U-Net Results:")

## Generate submission CSV

In [ ]:
import csv
import os
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
from PIL import Image
import torch
import numpy as np
import pandas as pd

submission_path = cfg.OUTPUT_DIR / 'submission_unet_final.csv'
results = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = [d for d in os.listdir(cfg.DATASET_DIR) if d.startswith('class_')]

for class_name in sorted(classes):
    ckpt_path = cfg.UNET_MODEL_DIR / class_name / 'best.pt'
    if not ckpt_path.exists(): continue

    model = unet.load_model(str(ckpt_path), device)
    test_dir = cfg.DATASET_DIR / class_name / 'test'
    stem_files = defaultdict(list)
    for fn in sorted(os.listdir(test_dir)):
        if fn.endswith('.png'):
            stem_files[unet._parse_view_and_id(fn)[0]].append(fn)

    for stem, fns in tqdm(stem_files.items(), desc=f'submit {class_name}'):
        view_results = []
        for fn in sorted(fns):
            _, view_id = unet._parse_view_and_id(fn)
            img = Image.open(str(test_dir / fn)).convert('RGB')
            orig_size = img.size
            img_rs = img.resize((cfg.UNET_IMAGE_SIZE, cfg.UNET_IMAGE_SIZE), Image.BILINEAR)
            fg = unet.foreground_from_image(img_rs)[0].numpy()
            v_t = torch.tensor([view_id], dtype=torch.long, device=device)
            prob = unet._infer_single_tta(model, img_rs, v_t, device, cfg.UNET_USE_TTA)
            prob = prob * fg
            view_results.append((fn, prob, orig_size))

        if cfg.UNET_MULTIVIEW_AGGREGATE:
            raw = unet._aggregate_views([r[1] for r in view_results])
            view_results = [(fn, p, sz) for (fn, _, sz), p in zip(view_results, raw)]

        for fn, prob, orig_size in view_results:
            prob_pil = Image.fromarray((prob * 255).astype(np.uint8)).resize(orig_size, Image.BILINEAR)
            prob_np = np.asarray(prob_pil, dtype=np.float32) / 255.0
            results.append({'ID': Path(fn).stem, 'Label': unet.float_matrix_to_q8rle(prob_np)})

print(f'Writing {len(results)} entries to {submission_path}')
with open(submission_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ID', 'Label'])
    w.writeheader()
    w.writerows(results)
print('Done.')